In [1]:
"""
__authors__ = Marc Vallvé Palerm, Ignasi Rebordosa Cots, Marc Salvatell Ferrerfàbrega
__date__    = Nov 2026
__title__   = Non-Linear Data Structures Project
"""

'\n__authors__ = Marc Vallvé Palerm, Ignasi Rebordosa Cots, Marc Salvatell Ferrerfàbrega\n__date__    = Nov 2026\n__title__   = Non-Linear Data Structures Project\n'

For this exercise, we decided to use an adjacency-list representation. Although a matrix-based representation would also be valid, it is less suitable for a graph with many sites and relatively few routes. For example, with 1,000 sites, a matrix would require a 1,000 × 1,000 structure, which means 1,000,000 cells and O(V²) initialization time, even though most pairs of sites are not connected. With an adjacency list, each site stores only its own data and its outgoing connections, so the structure uses O(V + E) memory, where V is the number of sites and E is the number of routes. Building the graph from the file also takes O(V + E) time on average, since a dictionary maps each site ID to its corresponding site object and allows each route endpoint to be found in O(1) average time. The adjacency list also naturally stores the connection attributes, such as destination, shipping time and restriction, and is more efficient for the traversals, topological sort and Dijkstra algorithm required later. A matrix would still be valid and would provide O(1) edge lookup, but it would waste space on the many non-existing routes.

In [2]:
#Product class
class Site:
    """Stores the data of one production site."""

    def __init__(self, site_id, product, corporation, material):
        self.id = site_id
        self.product = product
        self.corporation = corporation
        self.material = material
        self.connections = []  # List to store connections to other sites

    def get_class_info(self):
        return (
            f"Site(id={self.id}, product={self.product!r}, "
            f"corporation={self.corporation!r}, "
            f"material={self.material})"
        )

class Connection:
    """Stores the data of one connection between two production sites."""

    def __init__(self, origin, destination, shipping_time, restriction):
        self.origin = origin
        self.destination = destination
        self.shipping_time = shipping_time
        self.restriction = restriction

    def get_connection_info(self):
        return (
            f"Connection(origin={self.origin.id}, destination={self.destination.id}, "
            f"shipping_time={self.shipping_time}, restriction={self.restriction})"
        )


        

In [9]:
# ============================================================
# .paed FILE FORMAT
# ============================================================

# First line: number of sites/vertices
# Number of sites

# <site_id>;<product_name>;<corporation>;<site_material> (*Number of sites)

# Next line: number of routes/edges
# Number of routes

# <origin_id>;<destination_id>;<shipping_time>;<restriction> (*Number of routes)

file_name = r"Datasets/Graphs/aCityXS.paed"

#create a list to store the sites where each site will contain a class
sites = []
sites_by_id = {} #it will store side primary key: site object to access them faster direct access, O(1), instead of serial access, O(n)

with open(file_name, "r") as f:
    #read first line to get number of sites
    num_sites = int(f.readline().strip())

    #initialize a list to store the sites
    for i in range(num_sites):
        #read the line and split it by semicolon
        line = f.readline().strip()
        site_id, product, corporation, material = line.split(";")
        #create a Site object and append it to the list
        sites.append(Site(int(site_id), product, corporation, material))
        sites_by_id[int(site_id)] = sites[-1] #store the site object in the dictionary with its id as primary key
        
    #get how many routes there are
    num_routes = int(f.readline().strip())

    for i in range(num_routes):
        #read the line and split it by semicolon
        line = f.readline().strip()

        origin_id, destination_id, shipping_time, restriction = line.split(";")
        
        #create a Connection object and append it to the list of connections of the origin site
        #----------get the site where the connection starts and ends
        origin_site = sites_by_id[int(origin_id)]
        destination_site = sites_by_id[int(destination_id)]
        origin_site.connections.append(Connection(origin_site, destination_site, int(shipping_time), restriction))

In [10]:
#represent graph as adjacency list
for site in sites:
    print(f"Site {site.id} ({site.product}) has connections:")
    for connection in site.connections:
        print(f"  -> Site {connection.destination.id} ({connection.destination.product}), "
              f"Shipping Time: {connection.shipping_time}, Restriction: {connection.restriction}")


Site 67 (Triangular prism Catun No.2) has connections:
  -> Site 103 (Copper cuboid Mk II), Shipping Time: 19, Restriction: GLASS
  -> Site 118 (Copper hemisphere edge type 3), Shipping Time: 14, Restriction: NONE
Site 87 (Iron tetrahedron Mk I) has connections:
Site 119 (Fabric tetrahedron Catun No.1) has connections:
  -> Site 103 (Copper cuboid Mk II), Shipping Time: 24, Restriction: PLASTIC
  -> Site 121 (Wood cone mid-grade salvage), Shipping Time: 13, Restriction: GLASS
  -> Site 78 (Iron hexagonal prism edge type 1), Shipping Time: 22, Restriction: ORGANIC
Site 123 (Tinted glass icosahedron rusted junk) has connections:
  -> Site 103 (Copper cuboid Mk II), Shipping Time: 12, Restriction: PLASTIC
Site 101 (Tinted glass cube Mk I) has connections:
  -> Site 77 (Wax parallelepiped Mk II), Shipping Time: 19, Restriction: GLASS
  -> Site 134 (Polystyrene hexagonal prism Mk II), Shipping Time: 11, Restriction: METAL
Site 77 (Wax parallelepiped Mk II) has connections:
  -> Site 103 (Co

2.1.2.2: 
    
      A --> site ID entered by the user
            to get it from our code we will have to use sites_by_id[ID]
      A corporation can have more than 1 site
      We will have to create a tree that start with all conections of A, then the 2nd level are the connections of A, the 3rd the conections of the conections...
                                                                  A
                                                            _ _ _ | _ _ _
                                                            |           |
                                                            B           C
                                                           ...         ...
      To get all the nodes that are reachable from A and that have the same compar.. we will use DFS
      Since we will have to look through all nodes since theres not only 1 correct answer, we will use an O(n) time complexity

In [ ]:
# TODO 2.1.2.2:
# - Store the BFS depth/level for each matching site.
# - Display sites from deepest level to shallowest level.
# - Use ID ascending only as a tie-breaker.
# - Check if it works

ID_to_search = int(input("Enter the ID of the site to search for: "))

site_to_search = sites_by_id[ID_to_search]

visited = set() # Set to keep track of visited nodes during the search and do not repeat them
stack_corporation_nodes = [] # Stack of the result of the nodes that belong to the same corporation
corporation_to_search = site_to_search.corporation

#create recursive function to find all the nodes that belong to the same corporation
def find_corporation_nodes(site, corporation):

    #if its a site we already visited, we return to avoid infinite loops
    if site.id in visited:
        return
    
    visited.add(site.id)

    if site.corporation == corporation and site.id != ID_to_search: #we do not want to add the site we are searching for to the result
        stack_corporation_nodes.append(site)

    #loop through the connections of the site and call the function recursively for each connection
    for connection in site.connections:
        find_corporation_nodes(connection.destination, corporation)

find_corporation_nodes(site_to_search, corporation_to_search)

#return all the nodes that belong to the same corporation in ascending order
stack_corporation_nodes.sort(key=lambda site: site.id)

#display the result
for site in stack_corporation_nodes:
    print(f"Site {site.id} ({site.product}) belongs to the same corporation ({site.corporation})")
